# Would XGBoost (or other ML) improve the `FF1` soil-temperature gap-fill?

**A methods experiment, not a product.** It has no number because it exports nothing: it reads the
two screened `TS` files, reproduces the cross-depth gap-fill used by
[`10_METEO_TS_FF1_2004-2025.ipynb`](10_METEO_TS_FF1_2004-2025.ipynb), and compares it head-to-head
against multi-predictor linear regression, linear regression with thermal-memory features, gradient
boosting and XGBoost — on the held-out tests that matter, **including the extrapolation regime the
2010-2020 hole actually lives in**.

Run it to get the numbers; the closing section interprets them and gives a decision rule. It writes
no files.

## ℹ️ What this is testing, and why the answer is not obvious

The product fills a missing depth from the *other* depths of the same profile, by a per-calendar-
month linear regression fitted **separately for each sensor generation** (era-split), using the
**nearest** available depth as the single predictor. Validated held-out, that gives ~0.1-0.2 K where
a neighbouring depth is present and ~0.35-0.66 K for the deep depths predicted from the shallow ones
alone.

The question "would a stronger learner help?" has three parts, and they pull different ways:

1. **Most fills are already at the floor.** Adjacent depths correlate `r > 0.99`; a linear map
   between two depths 5-10 cm apart is near-perfect because heat conduction over that distance *is*
   nearly linear at 30-minute resolution. The residual is quantiser + genuine small-scale
   variability. No model can beat that — there is no headroom.
2. **The hard case has real structure a single-predictor line misses** — but the structure is
   mostly **thermal memory** (a deep temperature is a lagged, damped integral of the shallow
   history, not a function of the instantaneous shallow value) and **multiple predictors** (the
   whole shallow profile constrains the deep value better than one depth). Both are *feature*
   improvements that help a **linear** model just as much as a tree. So this experiment separates
   the gain from **features** (lags, multi-depth) from the gain from the **model class** (linear vs
   boosted trees).
3. **Extrapolation is the crux, and it cuts against trees.** Filling a decade-long hole means
   predicting the soil under conditions the training window may not bracket — a colder winter, a
   hotter summer. **Tree ensembles cannot extrapolate**: they output a constant beyond the training
   range, so they clamp extremes and bias systematically exactly where the fill is used. A linear
   model extrapolates linearly, which for a near-linear physical relationship is correct. The last
   test below measures this directly.

In [1]:
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor

try:
    import xgboost as xgb
    HAS_XGB = True
except Exception:
    HAS_XGB = False
    print("xgboost not installed - the XGBoost rows will be skipped. `uv add xgboost` to include them.")

warnings.filterwarnings("ignore")
pd.set_option("display.width", 1000)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 120)
print("sklearn present; xgboost:", HAS_XGB)

sklearn present; xgboost: True


## ⚙️ Settings and data

In [2]:
INDIR = r"F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\10_METEO\30_PRODUCTS"
DIIVE_FILE = "TS_FF1_SCREENED_30MIN_DIIVE_2020-2025"
MST_FILE = "TS_FF1_SCREENED_30MIN_MST_2004-2021"

DEPTHS = ["0.05", "0.1", "0.15", "0.2", "0.3", "0.5", "0.6"]

# The real channels behind each depth, exactly as the product resolves them (aliases and the
# mislabelled mst names excluded). Modern replicates in priority order; one early channel each.
MODERN = {"0.05": ["TS_FF1_0.05_2", "TS_FF1_0.05_3", "TS_FF1_0.05_4", "TS_FF1_0.05_1"],
          "0.1": ["TS_FF1_0.1_2", "TS_FF1_0.1_1"], "0.15": [],
          "0.2": ["TS_FF1_0.2_2", "TS_FF1_0.2_1"], "0.3": ["TS_FF1_0.3_2", "TS_FF1_0.3_1"],
          "0.5": ["TS_FF1_0.5_2", "TS_FF1_0.5_1"], "0.6": ["TS_FF1_0.6_1"]}
EARLY = {"0.05": "TS_PRF_FF1_0.05_1", "0.1": "TS_PRF_FF1_0.1_1", "0.15": "TS_PRF_FF1_0.15_1",
         "0.3": "TS_PRF_FF1_0.3_1", "0.5": "TS_PRF_FF1_0.5_1"}

ERA_BOUNDARY = pd.Timestamp("2020-04-10 14:45")   # first modern record
FILL_MIN_N = 500                                  # per-month min pairs for the linear baseline
RECORD_START, RECORD_END = "2004-09-07", "2025-12-31"

In [3]:
diive = pd.read_parquet(Path(INDIR) / f"{DIIVE_FILE}.parquet")
mst = pd.read_parquet(Path(INDIR) / f"{MST_FILE}.parquet")
full_index = pd.date_range(f"{RECORD_START} 00:15", f"{RECORD_END} 23:45", freq="30min",
                           name="TIMESTAMP_MIDDLE")


def merged_measured():
    """One measured series per depth on the export index: modern replicates primary-first, then the
    early channel spliced in with its overlap-measured offset (0.05/0.1) or at its own level."""
    M = pd.DataFrame(index=full_index)
    for d in DEPTHS:
        s = pd.Series(dtype=float)
        for c in MODERN[d]:
            v = diive[c].dropna()
            s = pd.concat([s, v.reindex(v.index.difference(s.index))]).sort_index()
        if d in EARLY:
            e = mst[EARLY[d]].dropna()
            ov = e.index.intersection(s.index)
            off = float((s.reindex(ov) - e.reindex(ov)).median()) if len(ov) > 2000 else 0.0
            e = e + off
            s = pd.concat([s, e.reindex(e.index.difference(s.index))]).sort_index()
        M[d] = s.reindex(full_index)
    return M


M = merged_measured()
IS_MODERN = pd.Series(full_index >= ERA_BOUNDARY, index=full_index)
print("measured coverage per depth [%]:")
print((100 * M.notna().mean()).round(1).to_dict())

measured coverage per depth [%]:
{'0.05': 94.2, '0.1': 94.0, '0.15': 72.1, '0.2': 26.4, '0.3': 50.0, '0.5': 50.0, '0.6': 26.4}


## 🧮 The five methods

All five predict one depth from the *others*. They differ in the predictors they see and the
function class that maps them:

| method | predictors | function | in the product? |
|---|---|---|---|
| **linear, nearest, monthly** | the single nearest available depth | one line per calendar month | **yes — this is the current fill** |
| linear, multi-depth | all other depths, concurrent | one linear map | no |
| linear, multi + memory | all other depths **plus their 3 h / 1 d / 10 d / 30 d / 120 d rolling means** | one linear map | no |
| gradient boosting | same features as "multi + memory" | boosted trees | no |
| XGBoost | same features | boosted trees | no |

The point of the middle two rows is to **attribute** any XGBoost win: if "linear, multi + memory"
already closes most of the gap, the gain was the *features*, not the model, and belongs in a linear
fill (cheap, transparent, extrapolates) rather than in a tree ensemble.

In [4]:
def usable_predictors(frame, candidates, train_mask, test_mask, min_cov=0.6):
    """Predictor depths that are actually present in BOTH the training and the test window.

    This is the detail that makes multi-depth gap-fill fiddly and is worth handling explicitly: the
    profile's depths have ragged availability (0.15 m ends 2021, 0.2/0.6 m start 2020), so a naive
    'use every other depth' feature matrix has an all-NaN column over any test window that predates
    or postdates one of them, and a row-wise dropna then empties the whole set. Here a candidate is
    kept only if it clears `min_cov` non-null in both masks."""
    keep = []
    for d in candidates:
        if (frame[d][train_mask].notna().mean() >= min_cov
                and frame[d][test_mask].notna().mean() >= min_cov):
            keep.append(d)
    return keep


def make_features(frame, predictor_depths, memory=False):
    """Concurrent predictor depths, optionally with rolling-mean 'thermal memory' features and a
    day-of-year harmonic. Windows are in 30MIN steps: 6=3h, 48=1d, 480=10d, 1440=30d, 5760=120d."""
    X = {d: frame[d] for d in predictor_depths}
    if memory:
        for d in predictor_depths:
            for w in (6, 48, 480, 1440, 5760):
                X[f"{d}_m{w}"] = frame[d].rolling(w, min_periods=int(0.5 * w)).mean()
    X = pd.DataFrame(X)
    doy = X.index.dayofyear.to_numpy()
    X["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
    X["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)
    return X


def scores(pred, truth):
    e = (pred - truth).dropna().to_numpy()
    if len(e) == 0:
        return {"n": 0, "bias": np.nan, "rmse": np.nan, "p95": np.nan}
    return {"n": len(e), "bias": round(float(e.mean()), 3), "rmse": round(float(np.sqrt((e ** 2).mean())), 3),
            "p95": round(float(np.percentile(np.abs(e), 95)), 3)}


def predict_nearest_monthly(frame, target, candidates, train_mask, test_mask):
    """The product's method: single nearest available predictor, one line per calendar month, per
    era. Reproduced here so the baseline in this notebook IS the baseline in the product."""
    others = sorted(candidates, key=lambda d: abs(float(d) - float(target)))
    pred = pd.Series(index=full_index, dtype=float)
    for era in (True, False):
        for p in others:
            need = test_mask & (IS_MODERN == era) & frame[p].notna() & pred.isna()
            if not need.any():
                continue
            fit = pd.DataFrame({"y": frame[target], "x": frame[p]})[train_mask & (IS_MODERN == era)].dropna()
            for mo in range(1, 13):
                dm = fit[fit.index.month == mo]
                if len(dm) < FILL_MIN_N:
                    continue
                b, a = np.polyfit(dm["x"].to_numpy(), dm["y"].to_numpy(), 1)
                sel = need & (full_index.month == mo)
                pred[sel] = a + b * frame.loc[sel, p]
    return pred


def predict_model(frame, model_fn, X, target, train_mask, test_mask):
    """Fit on the training rows where the features and target are all present, predict the test
    rows where the features are present. Returns (prediction on full_index, fit seconds)."""
    tr = pd.concat([X, frame[target].rename("y")], axis=1)[train_mask].dropna()
    te = X[test_mask].dropna()
    if len(tr) < 1000 or len(te) == 0:
        return pd.Series(index=full_index, dtype=float), np.nan
    t0 = time.perf_counter()
    m = model_fn().fit(tr.drop(columns="y"), tr["y"])
    fit_s = time.perf_counter() - t0
    pred = pd.Series(index=full_index, dtype=float)
    pred[te.index] = m.predict(te)
    return pred, fit_s


def MODEL_FNS():
    d = {"gradient boosting": lambda: GradientBoostingRegressor(
             n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.7)}
    if HAS_XGB:
        d["XGBoost"] = lambda: xgb.XGBRegressor(
            n_estimators=500, max_depth=4, learning_rate=0.05, subsample=0.7,
            colsample_bytree=0.8, n_jobs=4, verbosity=0)
    return d


def compare_methods(frame, target, candidates, train_mask, test_mask):
    """Run all five methods on one target and score them on their COMMON support - the timestamps
    every method managed to predict - so the RMSE columns are strictly comparable."""
    preds = {}
    preds["linear nearest monthly"] = predict_nearest_monthly(frame, target, candidates,
                                                              train_mask, test_mask)
    Xb = make_features(frame, candidates, memory=False)
    Xm = make_features(frame, candidates, memory=True)
    preds["linear multi"] = predict_model(frame, lambda: LinearRegression(), Xb, target,
                                          train_mask, test_mask)[0]
    preds["linear multi+memory"] = predict_model(frame, lambda: LinearRegression(), Xm, target,
                                                 train_mask, test_mask)[0]
    for name, fn in MODEL_FNS().items():
        preds[name] = predict_model(frame, fn, Xm, target, train_mask, test_mask)[0]

    truth = M[target].where(test_mask)
    common = truth.notna()
    for p in preds.values():
        common &= p.notna()
    out = {"n_common": int(common.sum())}
    for name, p in preds.items():
        out[name] = scores(p[common], truth[common])["rmse"]
    return out

## 🧪 Test 1 — leave-one-year-out (the easy regime: a neighbour is usually present)

Hide a whole year of a depth, fit on the rest, predict it back. This is the regime most fills are
in, and the expectation is that every method lands near the noise floor — a null result that is
worth showing, because it is where "just use XGBoost" quietly wastes effort.

In [5]:
def run_year(target, yr):
    hide = (full_index.year == yr) & M[target].notna()
    if hide.sum() < 3000:
        return None
    keep = ~hide
    cands = usable_predictors(M, [d for d in DEPTHS if d != target], keep, hide)
    if not cands:
        return None
    return {"depth_m": float(target), "year": yr, "predictors": ",".join(cands),
            **compare_methods(M, target, cands, keep, hide)}


TEST1 = pd.DataFrame([r for tgt in DEPTHS for yr in (2016, 2017, 2023)
                      if (r := run_year(tgt, yr)) is not None]).set_index(["depth_m", "year"])
print("Test 1 - leave-one-year-out, RMSE [K] (all methods scored on the same timestamps):")
display(TEST1.round(3))
_methcols = [c for c in TEST1.columns if c not in ("predictors", "n_common")]
print("\nColumn means (RMSE [K], lower is better):")
display(TEST1[_methcols].mean().round(3).to_frame("mean_rmse_K"))

KeyboardInterrupt: 

## 🧪 Test 2 — the real hole: predict the deep depths from the shallow ones alone

Hide **every** deep depth (0.2 / 0.3 / 0.5 / 0.6 m) from 2022 on and predict each from the shallow
depths only. This is the exact shape of the 2010-2020 hole the product fills, and the one place a
stronger learner has room to help.

In [ ]:
DEEP, SHALLOW = ["0.2", "0.3", "0.5", "0.6"], ["0.05", "0.1", "0.15"]
_hidewin = full_index >= "2022-01-01"


def run_hole(target):
    hide = _hidewin & M[target].notna()
    keep = ~_hidewin
    # Blind to ALL deep depths in the test window, exactly like the real 2010-2020 hole: only the
    # shallow depths are available to predict the deep ones. (0.15 m ends 2021, so in this test
    # window the usable shallow predictors are 0.05 and 0.1 - which is why the numbers here are
    # conservative relative to the real hole, where 0.15 m is still present.)
    Mblind = M.copy()
    Mblind.loc[_hidewin, DEEP] = np.nan
    cands = usable_predictors(Mblind, SHALLOW, keep, hide)
    return {"depth_m": float(target), "predictors": ",".join(cands),
            **compare_methods(Mblind, target, cands, keep, hide)}


TEST2 = pd.DataFrame([run_hole(t) for t in DEEP]).set_index("depth_m")
print("Test 2 - deep depths predicted from shallow only, RMSE [K]:")
display(TEST2.round(3))
_methcols = [c for c in TEST2.columns if c not in ("predictors", "n_common")]
print("\nColumn means (RMSE [K]):")
display(TEST2[_methcols].mean().round(3).to_frame("mean_rmse_K"))

## 🧪 Test 3 — the extrapolation hazard (this is where trees are supposed to fail)

Train on 2020-2023 and predict 2024-2025 at 0.5 m, then split the test by whether the day sits
**inside** the training temperature envelope (1st-99th percentile) or **outside** it. A linear model
should degrade gracefully outside; a tree ensemble should degrade sharply, because it cannot predict
a value beyond the range it was trained on — it clamps. If the "outside" bias for the trees is large
and one-signed, that is the argument against using them to fill a decade-long hole that surely
contains unseen extremes.

In [ ]:
_EXTRAP_TARGET = "0.5"
_extr_tr = (full_index >= "2020-04-11") & (full_index < "2024-01-01")
_extr_te = (full_index >= "2024-01-01")
# Predictors present across both windows (0.15 m has ended, so it drops out here).
_extr_cands = usable_predictors(M, [d for d in DEPTHS if d != _EXTRAP_TARGET], _extr_tr, _extr_te)
_extr_Xm = make_features(M, _extr_cands, memory=True)


def run_extrap(target=_EXTRAP_TARGET):
    lo, hi = M[target][_extr_tr].quantile(0.01), M[target][_extr_tr].quantile(0.99)
    outside = _extr_te & M[target].notna() & ((M[target] < lo) | (M[target] > hi))
    inside = _extr_te & M[target].notna() & ~outside
    rows = []
    fns = {"linear multi+memory": lambda: LinearRegression(), **MODEL_FNS()}
    for name, fn in fns.items():
        pred, _ = predict_model(M, fn, _extr_Xm, target, _extr_tr, _extr_te)
        for lbl, mask in [("inside envelope", inside), ("OUTSIDE envelope", outside)]:
            rows.append({"model": name, "regime": lbl, **scores(pred[mask], M[target][mask])})
    return pd.DataFrame(rows).set_index(["model", "regime"])


print(f"predictors usable for the extrapolation test: {_extr_cands}")
EXTRAP = run_extrap()
print(f"Test 3 - {_EXTRAP_TARGET} m, train 2020-2023, predict 2024-2025. Training envelope "
      f"[1..99 %]. RMSE and BIAS [K]:")
display(EXTRAP.round(3))
print("\nWatch the OUTSIDE-envelope bias: a large one-signed bias for the tree models is the")
print("clamping problem, and it is the regime a decade-long hole lives in.")

In [ ]:
# A picture of the same thing: predicted vs measured at 0.5 m in 2024-2025, coloured by regime.
target = _EXTRAP_TARGET
lo, hi = M[target][_extr_tr].quantile(0.01), M[target][_extr_tr].quantile(0.99)

fig, axs = plt.subplots(ncols=1 + len(MODEL_FNS()), figsize=(5.2 * (1 + len(MODEL_FNS())), 5),
                        dpi=100, layout="constrained")
axs = np.atleast_1d(axs)
_fns = {"linear multi+memory": lambda: LinearRegression(), **MODEL_FNS()}
for ax, (name, fn) in zip(axs, _fns.items()):
    pred, _ = predict_model(M, fn, _extr_Xm, target, _extr_tr, _extr_te)
    d = pd.DataFrame({"truth": M[target], "pred": pred})[_extr_te].dropna()
    inside = d["truth"].between(lo, hi)
    ax.scatter(d.loc[inside, "truth"], d.loc[inside, "pred"], s=3, alpha=.3, color="#00897B",
               label="inside envelope")
    ax.scatter(d.loc[~inside, "truth"], d.loc[~inside, "pred"], s=6, alpha=.5, color="#D32F2F",
               label="outside envelope")
    _lim = [d[["truth", "pred"]].min().min(), d[["truth", "pred"]].max().max()]
    ax.plot(_lim, _lim, color="#90A4AE", ls="--", lw=1)
    for _v in (lo, hi):
        ax.axvline(_v, color="#90A4AE", ls=":", lw=.8)
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("measured 0.5 m [°C]")
    ax.legend(fontsize=7)
axs[0].set_ylabel("predicted [°C]")
fig.suptitle("Predicted vs measured at 0.5 m, 2024-2025. Points outside the dotted training envelope "
             "are where trees clamp", fontsize=12)
plt.show()

## ⏱️ Cost, and the qualitative differences that do not show up in an RMSE

The numbers above are one axis; these are the others, and for a published, auditable product they
weigh as much as a tenth of a kelvin.

In [ ]:
_rows = []
for _name, _fn in {"linear multi+memory": lambda: LinearRegression(), **MODEL_FNS()}.items():
    _t0 = time.perf_counter()
    predict_model(M, _fn, _extr_Xm, _EXTRAP_TARGET, _extr_tr, _extr_te)
    _rows.append({"model": _name, "fit+predict_s": round(time.perf_counter() - _t0, 2)})
display(pd.DataFrame(_rows).set_index("model"))

print("""
Qualitative ledger (independent of the RMSE tables):

  transparency   linear: a per-month slope+intercept with physical units (a damping relation you
                 can print and sanity-check). trees: a black box; 'why is 2013 filled at 8.4 C' has
                 no inspectable answer.
  extrapolation  linear: extrapolates linearly, which is correct for a near-linear conduction
                 relation. trees: clamp to the training range - see Test 3.
  auditability   linear: negative controls and 'recover a known offset' guards, as elsewhere in this
                 repo. trees: hyperparameters, seeds, feature importances - harder to assert on.
  dependency     linear: numpy only. xgboost: a heavy compiled dependency added to a lean uv env.
  overfit risk   linear: none to speak of at this feature count. trees: real; needs held-out tuning
                 that itself has to be defended.
""")

## 🧭 Reading the result — the decision rule

Fill in from the tables above; the structure of the answer is fixed even before the numbers are:

1. **Test 1 (a neighbour present).** Expect every method within a few hundredths of a kelvin of the
   others, all near the quantiser floor. If so: for the ~90 % of fills that have an adjacent depth,
   **the model choice is irrelevant** and the simplest one wins by default.

2. **Test 2 (deep from shallow).** Compare three gaps:
   - *nearest-monthly → linear-multi → linear-multi+memory*: this is the gain from **features**
     (more predictors, thermal memory). It is available to the linear fill at zero extra dependency.
   - *linear-multi+memory → XGBoost*: this is the gain from the **model class** alone. If it is
     small (a few hundredths of a K), the ML win is really a feature win, and the right action is to
     **add multi-depth + memory features to the linear fill**, not to adopt XGBoost.
   - only if XGBoost beats the best linear by a **materially useful** margin (say > 0.15 K at a
     depth you care about) is there a case for it — and then only against Test 3.

3. **Test 3 (extrapolation) is a veto, not a tie-breaker.** A decade-long hole contains temperatures
   the training window did not, and a model that clamps them will bias the filled climate at exactly
   the extremes people study. If the trees show a large one-signed OUTSIDE-envelope bias, that
   **disqualifies them for this use** regardless of their in-sample RMSE.

**The most likely honest outcome**, given soil heat conduction is nearly linear and the dominant
missing signal is thermal lag: *the feature upgrade (multi-depth + memory) captures most of the
available gain and belongs in the linear fill; XGBoost adds little on top and loses on
extrapolation, transparency and dependency weight.* If your numbers say otherwise at a specific
depth, that is a targeted, defensible reason to use it **there** — not a reason to swap the whole
fill to a black box.

A concrete, low-regret upgrade this experiment may justify for the product: keep the fill **linear
and era-split**, but let it use **the two nearest available depths plus a 10-30 day rolling mean**
instead of a single instantaneous nearest depth. That is still inspectable, still extrapolates, and
picks up the thermal-memory signal that Test 2 isolates. Decide from the `linear multi+memory`
column whether it is worth the extra code.

In [ ]:
print("This notebook wrote nothing. It is an experiment feeding a decision, not a product.")
print("If the decision is 'upgrade the linear fill', the change lives in 10_METEO_TS_FF1's")
print("cross_depth_fill helper; if it is 'adopt a model', that is a larger discussion than a gap-fill.")